# House Price Prediction - Ames Housing Dataset
## Machine Learning Project - CAI2C08

**Project Goal**: Develop a machine learning model to predict house prices in Ames, Iowa based on various property features.

**Business Value**: Help buyers and sellers make informed pricing decisions, assist real estate agents with accurate valuations.

---

## 1. Setup and Data Loading

Import necessary libraries and load the dataset.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries imported successfully")

In [ ]:
# Load datasets
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"\n✓ Data loaded successfully")

## 2. Initial Data Exploration

**Purpose**: Understand the structure, types, and basic statistics of our dataset.

**Evaluation Criteria**: This section contributes to **Code Documentation (4%)** and prepares for **EDA (8%)**

In [ ]:
# Basic information
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

print(f"\nNumber of samples: {len(train_df):,}")
print(f"Number of features: {len(train_df.columns) - 1}")
print(f"Target variable: SalePrice")

# Check if meets project requirements (>1000 samples)
if len(train_df) >= 1000:
    print(f"\n✓ Dataset meets project requirement (≥1,000 samples)")
else:
    print(f"\n⚠ Dataset has fewer than 1,000 samples")

In [ ]:
# Display first few rows
print("\nFirst 5 rows of the dataset:")
train_df.head()

In [ ]:
# Data types and memory usage
print("\nData Types:")
train_df.info()

In [ ]:
# Statistical summary of numerical features
print("\nStatistical Summary of Numerical Features:")
train_df.describe()

In [ ]:
# Check for missing values
print("\nMissing Values Analysis:")
missing = train_df.isnull().sum()
missing_percent = 100 * missing / len(train_df)
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

print(f"\nTotal features with missing values: {len(missing_df)}")
print("\nTop 10 features with most missing values:")
print(missing_df.head(10))

In [ ]:
# Separate numerical and categorical features
numerical_features = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = train_df.select_dtypes(include=['object']).columns.tolist()

# Remove Id and SalePrice from numerical features
numerical_features = [f for f in numerical_features if f not in ['Id', 'SalePrice']]

print(f"\nNumerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total features (excluding Id and SalePrice): {len(numerical_features) + len(categorical_features)}")

In [ ]:
# Target variable (SalePrice) basic stats
print("\n" + "=" * 80)
print("TARGET VARIABLE ANALYSIS - SalePrice")
print("=" * 80)

print(f"\nMean: ${train_df['SalePrice'].mean():,.2f}")
print(f"Median: ${train_df['SalePrice'].median():,.2f}")
print(f"Std Dev: ${train_df['SalePrice'].std():,.2f}")
print(f"Min: ${train_df['SalePrice'].min():,.2f}")
print(f"Max: ${train_df['SalePrice'].max():,.2f}")

# Check for skewness
skewness = train_df['SalePrice'].skew()
print(f"\nSkewness: {skewness:.2f}")
if skewness > 0.5:
    print("→ Highly right-skewed distribution (log transformation may be needed)")
elif skewness < -0.5:
    print("→ Left-skewed distribution")
else:
    print("→ Approximately normal distribution")

## 3. Exploratory Data Analysis (EDA)

**Evaluation Criteria: 8% of total grade**

**Requirements for Grade A (≥80%):**
- Present insightful EDA with relevant visualizations (distribution, correlation, target vs features)
- Clearly labelled visualizations
- Interpret trends and outliers
- State implications for modeling

### 3.1 Target Variable Distribution

In [ ]:
# Visualization 1: SalePrice Distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Original distribution
axes[0].hist(train_df['SalePrice'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sale Price ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of House Prices (Original)', fontsize=14, fontweight='bold')
axes[0].axvline(train_df['SalePrice'].mean(), color='red', linestyle='--', label=f'Mean: ${train_df["SalePrice"].mean():,.0f}')
axes[0].axvline(train_df['SalePrice'].median(), color='green', linestyle='--', label=f'Median: ${train_df["SalePrice"].median():,.0f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Log-transformed distribution
log_prices = np.log1p(train_df['SalePrice'])
axes[1].hist(log_prices, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Log(Sale Price)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of House Prices (Log-Transformed)', fontsize=14, fontweight='bold')
axes[1].axvline(log_prices.mean(), color='red', linestyle='--', label=f'Mean: {log_prices.mean():.2f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Interpretation:")
print(f"The original price distribution is right-skewed (skewness: {skewness:.2f}).")
print(f"Log transformation reduces skewness to {log_prices.skew():.2f}, making it more normal.")
print(f"\n→ Implication: Using log(SalePrice) as target may improve model performance.")

### 3.2 Correlation Analysis

Identify which features are most strongly correlated with SalePrice.

In [ ]:
# Visualization 2: Correlation with SalePrice
correlations = train_df[numerical_features + ['SalePrice']].corr()['SalePrice'].sort_values(ascending=False)

# Top 15 correlations
top_correlations = correlations[1:16]  # Exclude SalePrice itself

plt.figure(figsize=(10, 8))
colors = ['green' if x > 0 else 'red' for x in top_correlations]
top_correlations.plot(kind='barh', color=colors, edgecolor='black')
plt.xlabel('Correlation with SalePrice', fontsize=12)
plt.title('Top 15 Features Correlated with House Price', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print(f"\nTop 5 positively correlated features:")
for i, (feature, corr) in enumerate(top_correlations.head(5).items(), 1):
    print(f"{i}. {feature}: {corr:.3f}")

print("\n→ Implication: These features should be prioritized in model training.")
print("  OverallQual, GrLivArea, and GarageCars show strong positive correlation.")

In [ ]:
# Visualization 3: Correlation Heatmap (Top features only)
top_features = correlations[1:11].index.tolist() + ['SalePrice']
corr_matrix = train_df[top_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap - Top 10 Features vs SalePrice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("Strong positive correlations (>0.6):")
strong_corr = corr_matrix[corr_matrix > 0.6].stack().reset_index()
strong_corr.columns = ['Feature1', 'Feature2', 'Correlation']
strong_corr = strong_corr[strong_corr['Feature1'] != strong_corr['Feature2']]
strong_corr = strong_corr[strong_corr['Feature1'] < strong_corr['Feature2']]  # Remove duplicates
print(strong_corr.to_string(index=False))
print("\n→ Implication: High multicollinearity detected between some features.")
print("  Consider removing one feature from highly correlated pairs.")

### 3.3 Key Features Analysis

Deep dive into the most important features.

In [ ]:
# Visualization 4: GrLivArea (Above Ground Living Area) vs SalePrice
plt.figure(figsize=(12, 6))
plt.scatter(train_df['GrLivArea'], train_df['SalePrice'], alpha=0.5, edgecolor='black')
plt.xlabel('Above Ground Living Area (sq ft)', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.title('Living Area vs House Price', fontsize=14, fontweight='bold')

# Add regression line
z = np.polyfit(train_df['GrLivArea'], train_df['SalePrice'], 1)
p = np.poly1d(z)
plt.plot(train_df['GrLivArea'], p(train_df['GrLivArea']), "r--", linewidth=2, label='Trend Line')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("Clear positive relationship: larger living area → higher price.")
print("\n⚠️ Outliers detected: Two houses with >4000 sq ft but low prices.")
print("→ Implication: These outliers may negatively impact model training and should be investigated.")

In [ ]:
# Visualization 5: OverallQual vs SalePrice (Box Plot)
plt.figure(figsize=(14, 6))
train_df.boxplot(column='SalePrice', by='OverallQual', figsize=(14, 6))
plt.suptitle('')  # Remove default title
plt.xlabel('Overall Quality Rating (1-10)', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.title('House Price Distribution by Overall Quality', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("Strong positive trend: Higher quality rating → significantly higher median price.")
print("Quality ratings 8-10 show larger price variance (wider boxes).")
print("\n→ Implication: OverallQual is a critical feature for prediction.")

In [ ]:
# Visualization 6: Categorical feature - Neighborhood vs SalePrice
neighborhood_prices = train_df.groupby('Neighborhood')['SalePrice'].median().sort_values()

plt.figure(figsize=(12, 8))
neighborhood_prices.plot(kind='barh', color='skyblue', edgecolor='black')
plt.xlabel('Median Sale Price ($)', fontsize=12)
plt.ylabel('Neighborhood', fontsize=12)
plt.title('Median House Price by Neighborhood', fontsize=14, fontweight='bold')
plt.axvline(x=train_df['SalePrice'].median(), color='red', linestyle='--', 
            label=f'Overall Median: ${train_df["SalePrice"].median():,.0f}')
plt.legend()
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print(f"Significant price variation across neighborhoods:")
print(f"- Highest: {neighborhood_prices.index[-1]} (${neighborhood_prices.iloc[-1]:,.0f})")
print(f"- Lowest: {neighborhood_prices.index[0]} (${neighborhood_prices.iloc[0]:,.0f})")
print(f"- Ratio: {neighborhood_prices.iloc[-1] / neighborhood_prices.iloc[0]:.1f}x difference")
print("\n→ Implication: Location (Neighborhood) is a crucial predictor.")

### 3.4 Outlier Detection

Identify and analyze potential outliers that may affect model performance.

In [ ]:
# Visualization 7: Outlier Detection using Z-score
from scipy.stats import zscore

# Calculate z-scores for SalePrice
z_scores = np.abs(zscore(train_df['SalePrice']))
outliers = train_df[z_scores > 3]

plt.figure(figsize=(14, 6))
plt.scatter(range(len(train_df)), train_df['SalePrice'], alpha=0.5, label='Normal')
plt.scatter(outliers.index, outliers['SalePrice'], color='red', s=100, 
            label=f'Outliers (Z-score > 3): {len(outliers)}')
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.title('Outlier Detection in House Prices', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Interpretation:")
print(f"Detected {len(outliers)} extreme outliers (Z-score > 3) in SalePrice.")
print(f"\nOutlier properties:")
print(outliers[['GrLivArea', 'OverallQual', 'SalePrice']].to_string())
print("\n→ Implication: Consider removing these outliers or using robust models.")

### 3.5 Missing Values Visualization

In [ ]:
# Visualization 8: Missing Values Heatmap
features_with_missing = missing_df.head(15).index

plt.figure(figsize=(12, 8))
sns.heatmap(train_df[features_with_missing].isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Values Pattern (Top 15 Features)', fontsize=14, fontweight='bold')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("Features like PoolQC, MiscFeature, and Alley have >90% missing values.")
print("→ Implication: These features may need to be dropped rather than imputed.")

### 3.6 Feature Distribution Analysis

In [ ]:
# Visualization 9: Distribution of key numerical features
key_numerical = ['GrLivArea', 'TotalBsmtSF', 'GarageArea', 'YearBuilt']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, feature in enumerate(key_numerical):
    axes[idx].hist(train_df[feature], bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_xlabel(feature, fontsize=11)
    axes[idx].set_ylabel('Frequency', fontsize=11)
    axes[idx].set_title(f'Distribution of {feature}', fontsize=12, fontweight='bold')
    axes[idx].axvline(train_df[feature].mean(), color='red', linestyle='--', 
                     label=f'Mean: {train_df[feature].mean():.1f}')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("- GrLivArea: Right-skewed, most houses 1000-2000 sq ft")
print("- TotalBsmtSF: Many zeros (houses without basements)")
print("- GarageArea: Similar to basement, many zeros")
print("- YearBuilt: Peak around 2000s, representing newer homes")
print("\n→ Implication: Skewed features may benefit from transformation.")

### 3.7 Year Built vs Price (Time Trend)

In [ ]:
# Visualization 10: Year Built vs SalePrice
year_price = train_df.groupby('YearBuilt')['SalePrice'].mean()

plt.figure(figsize=(14, 6))
plt.plot(year_price.index, year_price.values, marker='o', linewidth=2, markersize=4)
plt.xlabel('Year Built', fontsize=12)
plt.ylabel('Average Sale Price ($)', fontsize=12)
plt.title('House Price Trend Over Construction Year', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("Clear upward trend: Newer homes command higher prices.")
print("Significant price jump for homes built after 2000.")
print("\n→ Implication: Creating 'HouseAge' feature may capture depreciation effect.")

### 3.8 Summary of EDA Findings

**Key Insights:**

1. **Target Variable (SalePrice)**:
   - Right-skewed distribution (skewness: ~1.88)
   - Log transformation recommended for modeling
   - Price range: $34,900 to $755,000

2. **Most Important Features**:
   - OverallQual (r=0.79): Strongest predictor
   - GrLivArea (r=0.71): Living area size
   - GarageCars (r=0.64): Garage capacity
   - GarageArea (r=0.62): Garage size
   - TotalBsmtSF (r=0.61): Basement area

3. **Data Quality Issues**:
   - 19 features with missing values
   - Features with >80% missing: PoolQC, MiscFeature, Alley, Fence → Consider dropping
   - ~4 extreme outliers detected in SalePrice

4. **Feature Characteristics**:
   - High multicollinearity between garage-related features
   - Neighborhood significantly affects price (3.5x difference)
   - Many features are right-skewed (GrLivArea, TotalBsmtSF)

**Implications for Modeling:**
- ✅ Use log(SalePrice) as target variable
- ✅ Apply feature engineering: HouseAge, TotalArea, Quality×Area interactions
- ✅ Handle missing values strategically (drop vs impute)
- ✅ Consider removing multicollinear features
- ✅ Apply transformation to skewed numerical features
- ✅ One-hot encode categorical features (esp. Neighborhood)
- ⚠️ Outlier removal or robust modeling techniques needed

---

## Next Steps:

1. ✅ Data Cleaning and Preprocessing
2. ✅ Feature Engineering
3. ✅ Model Training and Comparison
4. ✅ Hyperparameter Tuning
5. ✅ Model Evaluation and Selection
6. ✅ Deployment (Streamlit App)

---

**Version Control Note**: Remember to commit this notebook after completing EDA.